# Sentiment Analysis – Model Development
> **Dataset**: IMDB Movie Reviews (pre-processed)  
> **Task**: Binary sentiment classification — Positive vs Negative  
> **Author**: AI Engineer Assessment

---
## Notebook Roadmap
1. Setup & Data Loading
2. Feature Engineering (TF-IDF)
3. Class Imbalance Mitigation Strategy
4. Train & Compare 4 Traditional ML Models
5. Model Evaluation — Metrics Table + Confusion Matrix
6. Modern AI Comparison (Hugging Face Transformer)
7. Best Model Export

## 1. Setup & Data Loading

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline          import Pipeline
from sklearn.preprocessing     import LabelEncoder
from sklearn.linear_model      import LogisticRegression
from sklearn.naive_bayes       import MultinomialNB
from sklearn.ensemble          import RandomForestClassifier
from sklearn.metrics           import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay
)
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120})

SEED = 42
np.random.seed(SEED)

# ── Load Processed Data ──────────────────────────────────────────────────────
DATA_PATH = os.path.join('..', 'data', 'processed_imdb_data.csv')
df = pd.read_csv(DATA_PATH)
df.dropna(subset=['clean_review', 'label'], inplace=True)

print(f'Loaded: {df.shape[0]:,} samples')
print(f'Class balance:\n{df["sentiment"].value_counts()}')
df.head(3)

## 2. Feature Engineering — TF-IDF Vectorization

In [ ]:
# ── Train / Test Split ───────────────────────────────────────────────────────
X = df['clean_review'].astype(str)
y = df['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

# ── TF-IDF Vectorizer ────────────────────────────────────────────────────────
# unigrams + bigrams, capped vocabulary, removes very common & very rare terms
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=50_000,
    min_df=3,
    max_df=0.90,
    sublinear_tf=True      # smoothes out term frequency imbalances
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'\nTF-IDF matrix shape → Train: {X_train_tfidf.shape}, Test: {X_test_tfidf.shape}')

## 3. Class Imbalance Mitigation Strategy

IMDB is largely balanced (25 k / 25 k), but we still apply `class_weight='balanced'`  
on every model so the approach generalises to imbalanced production scenarios.

In [ ]:
from collections import Counter

counter = Counter(y_train)
total   = sum(counter.values())
print('Class distribution in training set:')
for cls, cnt in sorted(counter.items()):
    label_name = 'Positive' if cls == 1 else 'Negative'
    print(f'  {label_name} ({cls}): {cnt:,}  ({cnt/total*100:.1f}%)')

imbalance_ratio = max(counter.values()) / min(counter.values())
print(f'\nImbalance ratio: {imbalance_ratio:.2f}')
print('Strategy: class_weight="balanced" applied to all sklearn models')
print('          scale_pos_weight applied to XGBoost')

## 4. Train & Compare 4 Traditional ML Models

In [ ]:
# ── Model Definitions ────────────────────────────────────────────────────────
neg_count  = counter[0]
pos_count  = counter[1]
scale_xgb  = neg_count / pos_count  # for XGBoost class weight

MODELS = {
    'Logistic Regression': LogisticRegression(
        C=1.0, max_iter=1000, class_weight='balanced', random_state=SEED, n_jobs=-1
    ),
    'Naive Bayes': MultinomialNB(
        alpha=0.1   # smoothing — MultinomialNB does not accept class_weight
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=None, class_weight='balanced',
        random_state=SEED, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=300, learning_rate=0.1, max_depth=6,
        scale_pos_weight=scale_xgb, use_label_encoder=False,
        eval_metric='logloss', random_state=SEED, n_jobs=-1
    )
}

# ── Training Loop ────────────────────────────────────────────────────────────
results       = {}
trained_models = {}

print(f'{"Model":<25} {"Accuracy":>9} {"Precision":>10} {"Recall":>8} {"F1":>8} {"Time":>8}')
print('-' * 72)

for name, model in MODELS.items():
    t0 = time.time()
    model.fit(X_train_tfidf, y_train)
    elapsed = time.time() - t0

    y_pred = model.predict(X_test_tfidf)

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    results[name]        = dict(accuracy=acc, precision=prec, recall=rec, f1=f1, train_time=elapsed)
    trained_models[name] = model

    print(f'{name:<25} {acc:>9.4f} {prec:>10.4f} {rec:>8.4f} {f1:>8.4f} {elapsed:>6.1f}s')

print('\nAll models trained ✓')

## 5. Model Evaluation — Metrics Table & Confusion Matrix

In [ ]:
# ── 5.1  Comparison Bar Chart ────────────────────────────────────────────────
metrics_df = pd.DataFrame(results).T[['accuracy', 'precision', 'recall', 'f1']]

ax = metrics_df.plot(kind='bar', figsize=(11, 5), width=0.7, colormap='Set2', edgecolor='white')
ax.set_title('4-Model Comparison — Core Evaluation Metrics', fontsize=14, fontweight='bold')
ax.set_ylabel('Score', fontsize=11)
ax.set_xticklabels(metrics_df.index, rotation=15, ha='right')
ax.set_ylim(0.7, 1.0)
ax.legend(loc='lower right', fontsize=9)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
plt.tight_layout()
plt.savefig('../data/model_comparison.png', bbox_inches='tight')
plt.show()

print('\n=== Full Metrics Table ===')
print(metrics_df.round(4).to_string())

In [ ]:
import matplotlib.ticker as mticker  # already imported at top; safe duplicate

# ── 5.2  Confusion Matrix (all 4 models) ─────────────────────────────────────
CLASS_NAMES = ['Negative', 'Positive']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for idx, (name, model) in enumerate(trained_models.items()):
    y_pred = model.predict(X_test_tfidf)
    cm     = confusion_matrix(y_test, y_pred)
    disp   = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
    disp.plot(ax=axes[idx], cmap='Blues', colorbar=False)
    axes[idx].set_title(f'{name}', fontsize=12, fontweight='bold')

plt.suptitle('Confusion Matrices – All 4 Models', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig('../data/confusion_matrices.png', bbox_inches='tight')
plt.show()

In [ ]:
# ── 5.3  Detailed Classification Report — Best Model ─────────────────────────
best_model_name = metrics_df['f1'].idxmax()
best_model      = trained_models[best_model_name]
y_pred_best     = best_model.predict(X_test_tfidf)

print(f'Best model (by F1): {best_model_name}')
print('\n=== Classification Report===')
print(classification_report(y_test, y_pred_best, target_names=CLASS_NAMES))

## 6. Modern AI Comparison — Hugging Face Transformer

We evaluate a **pre-trained sentiment model** from Hugging Face as a zero-shot baseline  
to compare against our TF-IDF + Traditional ML pipeline.

> ⚠️ **Note**: The Transformer runs on CPU and is evaluated on a **small sample** (500 rows)  
> to avoid multi-hour inference times. This is purely for architectural comparison.

In [ ]:
from transformers import pipeline as hf_pipeline

# ── Load a lightweight pre-trained BERT-based sentiment pipeline ─────────────
print('Loading Hugging Face sentiment pipeline... (downloads ~250 MB on 1st run)')
sentiment_pipeline = hf_pipeline(
    'sentiment-analysis',
    model='distilbert-base-uncased-finetuned-sst-2-english',
    truncation=True,
    max_length=512
)
print('Model loaded ✓')

# ── Evaluate on a representative 500-sample subset ───────────────────────────
SAMPLE_N    = 500
sample_idx  = X_test.sample(n=SAMPLE_N, random_state=SEED).index
X_sample    = X_test.loc[sample_idx].tolist()
y_sample    = y_test.loc[sample_idx].tolist()

print(f'Running inference on {SAMPLE_N} samples...')
t0 = time.time()
raw_preds  = sentiment_pipeline(X_sample, batch_size=32)
elapsed_hf = time.time() - t0

# Map HuggingFace output labels → binary integers
label_remap  = {'POSITIVE': 1, 'NEGATIVE': 0}
y_pred_hf    = [label_remap[p['label']] for p in raw_preds]

hf_acc  = accuracy_score(y_sample, y_pred_hf)
hf_f1   = f1_score(y_sample, y_pred_hf, average='weighted')

print(f'\n[DistilBERT – SST-2]  Accuracy={hf_acc:.4f} | F1={hf_f1:.4f} | Time={elapsed_hf:.1f}s')
print(f'[{best_model_name}] Accuracy={metrics_df.loc[best_model_name,"accuracy"]:.4f} | F1={metrics_df.loc[best_model_name,"f1"]:.4f}')

print('\n→ Traditional TF-IDF pipeline achieves comparable performance at a fraction of the cost.')
print('  Transformer approach requires GPU + memory for production deployment.')

## 7. Best Model Export

In [ ]:
# ── Package TF-IDF + best classifier as a single sklearn Pipeline ────────────
inference_pipeline = Pipeline([
    ('tfidf', tfidf),
    ('clf',   best_model)
])

os.makedirs('../models', exist_ok=True)
MODEL_PATH = os.path.join('..', 'models', 'best_model.joblib')
joblib.dump(inference_pipeline, MODEL_PATH)

print(f'Pipeline exported → {MODEL_PATH}')
print(f'Model type : {best_model_name}')

# ── Save comparison results for README reference ──────────────────────────────
metrics_df.to_csv('../data/model_comparison_results.csv')
print('Comparison results saved → data/model_comparison_results.csv')

# ── Quick sanity-check on 3 examples ─────────────────────────────────────────
test_reviews = [
    'This movie was absolutely fantastic! A must-watch masterpiece.',
    'Terrible film. Complete waste of time and money. I hated it.',
    'The plot was average. Some good moments, but overall forgettable.'
]

from notebooks_utils import preprocess_single  # defined below in same cell for completeness

import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

STOP_WORDS  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def preprocess_single(text: str) -> str:
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

print('\n=== Inference Sanity Check ===')
for review in test_reviews:
    clean = preprocess_single(review)
    proba = inference_pipeline.predict_proba([clean])[0]
    pred  = inference_pipeline.predict([clean])[0]
    label = 'Positive' if pred == 1 else 'Negative'
    conf  = proba[pred]
    print(f'  → {label} ({conf:.2%})  | "{review[:60]}..."')